# Single-Class YOLO Training — Microplastic Detection (Kaggle)

> **Pipeline: YOLO (detect "microplastic") → EfficientNet (classify fiber/film/fragment)**

## Overview

This notebook trains a **single-class YOLOv8m** detector that learns to find all microplastic
particles regardless of type. Classification into fiber/film/fragment is handled downstream
by EfficientNet. This two-stage approach improves both detection recall and classification accuracy.

## Kaggle Setup

### Before Running This Notebook

1. **Upload your dataset as a Kaggle Dataset:**
   - Go to [kaggle.com/datasets](https://www.kaggle.com/datasets) → **New Dataset**
   - Upload the `yolo_augmented_single/` folder (with `dataset.yaml`, `images/`, `labels/`)
   - Name it e.g. `mp-yolo-augmented-single`

2. **Add the dataset to this notebook:**
   - Click **Add Data** (right sidebar) → search your dataset → **Add**
   - It will appear at `/kaggle/input/mp-yolo-augmented-single/`

3. **Enable GPU:**
   - Settings → Accelerator → **GPU T4 x2** or **GPU P100**

4. **Enable Internet:**
   - Settings → Internet → **On** (needed for pip install & pretrained weights)

## Training Configuration

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| Model | YOLOv8m | Best accuracy/VRAM trade-off for T4 |
| Image size | 1280 | High resolution for small objects |
| Batch size | 2 | Maximum for YOLOv8m @ 1280 on T4 (15 GB) |
| Epochs | 200 | With early stopping (patience=50) |
| Optimizer | AdamW | Better generalisation on small datasets |
| Classes | 1 (microplastic) | Single-class detection |
| Cache | Disk | Saves RAM; uses local SSD for fast I/O |
| AMP | Enabled | ~2× throughput, halves activation memory |
| nbs | 64 | Gradient accumulation compensates small batch |

## Kaggle Directory Structure

```
/kaggle/input/mp-yolo-augmented-single/   ← YOUR UPLOADED DATASET (read-only)
├── dataset.yaml
├── images/
│   ├── train/   (800 images)
│   └── val/     (200 images)
└── labels/
    ├── train/
    └── val/

/kaggle/working/                           ← WRITABLE OUTPUT DIRECTORY
├── dataset/                                ← local copy for training
└── experiments/yolo/                       ← trained weights saved here
    └── mp_yolov8m_single_class/
        └── weights/
            ├── best.pt                     ← DOWNLOAD THIS
            └── last.pt
```

## Where to Find Trained Models

After training completes:
1. Go to the **Output** tab of your notebook
2. Download `experiments/yolo/mp_yolov8m_single_class/weights/best.pt`
3. Or click **Save Version** → the output is saved permanently as a **Kaggle Dataset**
   that you can attach to other notebooks

## 1. Environment Setup

In [ ]:
# ==============================================================================
# 1. ENVIRONMENT SETUP
# Install dependencies (Kaggle needs internet enabled).
# ==============================================================================

!pip install ultralytics>=8.1.0 --quiet

import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")
print("Environment setup complete.")

## 2. GPU Verification & Reproducibility

**Kaggle GPU Options:**

| GPU | VRAM | Configuration |
|-----|------|---------------|
| T4 x2 | 15 GB each | Use `device=0`, batch=2, imgsz=1280 |
| P100 | 16 GB | Same config works, slight extra headroom |

**Kaggle Free Tier Limits:**
- 30 hours GPU per week
- 12-hour max session
- `/kaggle/working/` persists up to 20 GB after save

In [ ]:
# ==============================================================================
# 2. GPU VERIFICATION & REPRODUCIBILITY
# ==============================================================================

import torch
import random
import numpy as np
import os

assert torch.cuda.is_available(), (
    "No GPU detected. Go to Settings > Accelerator > GPU T4 x2 or GPU P100."
)

gpu_name   = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU           : {gpu_name}")
print(f"VRAM          : {gpu_mem_gb:.1f} GB")
print(f"PyTorch       : {torch.__version__}")
print(f"CUDA          : {torch.version.cuda}")

# Check available RAM
import psutil
ram_gb = psutil.virtual_memory().total / 1e9
ram_avail_gb = psutil.virtual_memory().available / 1e9
print(f"RAM total     : {ram_gb:.1f} GB")
print(f"RAM available : {ram_avail_gb:.1f} GB")

# Check local disk space
disk = psutil.disk_usage('/kaggle/working')
print(f"Disk total    : {disk.total / 1e9:.1f} GB")
print(f"Disk free     : {disk.free / 1e9:.1f} GB")

# Deterministic seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print(f"\nSeed: {SEED}")
print("GPU verification complete.")

## 3. Dataset: Copy to Working Directory

Kaggle datasets at `/kaggle/input/` are **read-only**. We copy to `/kaggle/working/dataset/`
so YOLO can write cache files alongside the images.

**IMPORTANT:** Change `KAGGLE_DATASET_NAME` below to match your uploaded dataset name.

**Expected structure at `/kaggle/input/<your-dataset-name>/`:**
```
dataset.yaml
images/
  train/   (800 images)
  val/     (200 images)
labels/
  train/
  val/
```

In [ ]:
# ==============================================================================
# 3. COPY DATASET TO WORKING DIRECTORY
# /kaggle/input/ is read-only, so we copy to /kaggle/working/
# ==============================================================================

import os
import shutil
import yaml
from pathlib import Path

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
LOCAL_DATASET_PATH = "/kaggle/working/dataset"
OUTPUT_PATH        = "/kaggle/working/experiments/yolo"

os.makedirs(OUTPUT_PATH, exist_ok=True)

# ---------------------------------------------------------------------------
# Auto-detect dataset location under /kaggle/input/
# Kaggle mount names can differ from the dataset display name.
# We search for dataset.yaml anywhere under /kaggle/input/.
# ---------------------------------------------------------------------------
INPUT_ROOT = Path("/kaggle/input")
print(f"Contents of /kaggle/input/: {os.listdir(INPUT_ROOT)}")

# Search for dataset.yaml
candidates = list(INPUT_ROOT.rglob("dataset.yaml"))
assert candidates, (
    f"No dataset.yaml found anywhere under {INPUT_ROOT}\n"
    f"Top-level folders: {os.listdir(INPUT_ROOT)}\n"
    f"Make sure you uploaded a YOLO dataset containing dataset.yaml"
)

# Pick the first match (or the one with 'single' in the path if multiple)
if len(candidates) == 1:
    SRC_PATH = str(candidates[0].parent)
else:
    # Prefer the one with 'single' in the path
    single = [c for c in candidates if 'single' in str(c).lower()]
    SRC_PATH = str(single[0].parent) if single else str(candidates[0].parent)
    print(f"Multiple dataset.yaml found: {[str(c) for c in candidates]}")

print(f"Auto-detected dataset at: {SRC_PATH}")

# Verify expected structure
src = Path(SRC_PATH)
assert (src / "images").exists(), f"No images/ folder at {SRC_PATH}"
assert (src / "labels").exists(), f"No labels/ folder at {SRC_PATH}"

# ---------------------------------------------------------------------------
# Copy to working directory
# ---------------------------------------------------------------------------
if not os.path.exists(LOCAL_DATASET_PATH):
    print("Copying dataset to working directory...")
    shutil.copytree(SRC_PATH, LOCAL_DATASET_PATH)
    print("Copy complete.")
else:
    print(f"Dataset already exists at {LOCAL_DATASET_PATH}")

DATASET_PATH = LOCAL_DATASET_PATH
YAML_PATH    = f"{DATASET_PATH}/dataset.yaml"

# ---------------------------------------------------------------------------
# Fix dataset.yaml path for Kaggle
# ---------------------------------------------------------------------------
assert os.path.exists(YAML_PATH), f"dataset.yaml not found at {YAML_PATH}"

with open(YAML_PATH) as f:
    ds_config = yaml.safe_load(f)

if ds_config.get("path") != DATASET_PATH:
    print(f"[FIX] Updating dataset.yaml 'path':")
    print(f"       Old: {ds_config.get('path')}")
    print(f"       New: {DATASET_PATH}")
    ds_config["path"] = DATASET_PATH
    with open(YAML_PATH, "w") as f:
        yaml.dump(ds_config, f, default_flow_style=False)

print("\ndataset.yaml contents:")
print(yaml.dump(ds_config, default_flow_style=False))

## 3.1 Dataset Verification

In [ ]:
# ==============================================================================
# 3.1 DATASET VERIFICATION
# Count images/labels, detect mismatches, show label distribution.
# ==============================================================================

from pathlib import Path
from collections import Counter
import numpy as np

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

for split in ["train", "val"]:
    img_dir = Path(DATASET_PATH) / "images" / split
    lbl_dir = Path(DATASET_PATH) / "labels" / split

    assert img_dir.exists(), f"Missing directory: {img_dir}"
    assert lbl_dir.exists(), f"Missing directory: {lbl_dir}"

    images = {f.stem for f in img_dir.iterdir() if f.suffix.lower() in IMG_EXTS}
    labels = {f.stem for f in lbl_dir.iterdir() if f.suffix == ".txt"}

    missing_labels = images - labels
    orphan_labels  = labels - images

    total_objects = 0
    bbox_widths, bbox_heights = [], []
    for lbl_file in lbl_dir.glob("*.txt"):
        for line in lbl_file.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) >= 5:
                total_objects += 1
                bbox_widths.append(float(parts[3]))
                bbox_heights.append(float(parts[4]))

    print(f"[{split:5s}]  images: {len(images):4d}  |  labels: {len(labels):4d}  |  "
          f"objects: {total_objects:5d}  |  "
          f"missing labels: {len(missing_labels)}  |  orphan labels: {len(orphan_labels)}")

    if bbox_widths:
        w_arr = np.array(bbox_widths)
        h_arr = np.array(bbox_heights)
        IMGSZ_REF = 1280
        print(f"  Bbox size (px @ {IMGSZ_REF}):")
        print(f"    Median : {np.median(w_arr)*IMGSZ_REF:.0f} x {np.median(h_arr)*IMGSZ_REF:.0f}")
        print(f"    Mean   : {np.mean(w_arr)*IMGSZ_REF:.0f} x {np.mean(h_arr)*IMGSZ_REF:.0f}")
        small_count = np.sum((w_arr * IMGSZ_REF < 32) & (h_arr * IMGSZ_REF < 32))
        print(f"    Objects < 32x32 px : {small_count} / {len(w_arr)} "
              f"({100 * small_count / len(w_arr):.1f}%)")

    if missing_labels:
        print(f"  WARNING: {len(missing_labels)} images have no matching label file.")

print("\nDataset verification complete.")

## 4. Model Training

### Key Memory-Saving Strategies

1. **`cache='disk'`**: Decoded images stored on local SSD instead of RAM.
2. **`batch=2` + `nbs=64`**: Small batch to fit VRAM, but YOLO auto-accumulates gradients
   as if batch=64 (accumulate = nbs/batch = 32 steps).
3. **`amp=True`**: Mixed precision (FP16) halves memory for activations and weights.
4. **`workers=2`**: Kaggle typically has 4 vCPUs but 2 workers is safe.
5. **`multi_scale=False`**: Dynamic resolution changes cause unpredictable VRAM spikes at 1280.
6. **`save_period=25`**: Checkpoints every 25 epochs.

### Single-Class Advantages

- Simpler task → converges faster with less data
- No inter-class confusion (fiber vs film distinction is hard for YOLO)
- Higher recall — model focuses purely on "is this a microplastic?"
- Classification handled by EfficientNet on cropped regions (much easier task)

In [ ]:
# ==============================================================================
# 4. MODEL TRAINING — Single-Class YOLOv8m
# Optimised for Kaggle T4/P100 with disk caching.
# ==============================================================================

from ultralytics import YOLO
from pathlib import Path
import shutil, datetime

# ---------------------------------------------------------------------------
# Core hyperparameters
# ---------------------------------------------------------------------------
MODEL          = "yolov8m.pt"
IMGSZ          = 1280
BATCH_SIZE     = 2
EPOCHS         = 200
PATIENCE       = 50
EXPERIMENT     = "mp_yolov8m_single_class"

# ---------------------------------------------------------------------------
# Optimizer
# ---------------------------------------------------------------------------
OPTIMIZER      = "AdamW"
LR0            = 0.0005
LRF            = 0.01
WEIGHT_DECAY   = 5e-4
WARMUP_EPOCHS  = 5.0
WARMUP_MOM     = 0.8
WARMUP_BIAS_LR = 0.1

# ---------------------------------------------------------------------------
# Augmentation
# ---------------------------------------------------------------------------
MOSAIC         = 1.0
COPY_PASTE     = 0.3
MIXUP          = 0.2
HSV_H          = 0.015
HSV_S          = 0.7
HSV_V          = 0.4
DEGREES        = 15.0
TRANSLATE      = 0.2
SCALE          = 0.5
SHEAR          = 5.0
PERSPECTIVE    = 0.0005
FLIPUD         = 0.5
FLIPLR         = 0.5
ERASING        = 0.4
CLOSE_MOSAIC   = 20
LABEL_SMOOTH   = 0.0

# ---------------------------------------------------------------------------
# Loss weights
# ---------------------------------------------------------------------------
BOX_LOSS       = 7.5
CLS_LOSS       = 0.5
DFL_LOSS       = 1.5

# ---------------------------------------------------------------------------
# Auto-backup callback — copies weights every BACKUP_EVERY epochs
# Backups go to /kaggle/working/backups/epoch_<N>/ and are always accessible
# in the Output tab even without a manual Save Version.
# ---------------------------------------------------------------------------
BACKUP_EVERY = 50
BACKUP_ROOT  = Path("/kaggle/working/backups")

def auto_backup(trainer):
    epoch = trainer.epoch + 1          # 1-based
    if epoch % BACKUP_EVERY != 0:
        return

    weights_dir = Path(trainer.save_dir) / "weights"
    backup_dir  = BACKUP_ROOT / f"epoch_{epoch:04d}"
    backup_dir.mkdir(parents=True, exist_ok=True)

    for name in ("best.pt", "last.pt"):
        src = weights_dir / name
        if src.exists():
            shutil.copy2(src, backup_dir / name)

    ts = datetime.datetime.now().strftime("%H:%M:%S")
    print(f"\n[AUTO-BACKUP] epoch {epoch} → {backup_dir}  ({ts})\n")

# ---------------------------------------------------------------------------
# Load model & register callback
# ---------------------------------------------------------------------------
model = YOLO(MODEL)
model.add_callback("on_train_epoch_end", auto_backup)

print(f"{'='*70}")
print(f"  Single-Class YOLO Training — Microplastic Detection (Kaggle)")
print(f"{'='*70}")
print(f"  Model        : {MODEL}")
print(f"  Classes      : 1 (microplastic)")
print(f"  Image size   : {IMGSZ}")
print(f"  Batch size   : {BATCH_SIZE} (nbs=64 → accumulate={64//BATCH_SIZE} steps)")
print(f"  Epochs       : {EPOCHS} (early stopping patience={PATIENCE})")
print(f"  Optimizer    : {OPTIMIZER}, lr0={LR0}, cos_lr=True")
print(f"  Cache        : disk")
print(f"  AMP          : Enabled")
print(f"  Auto-backup  : every {BACKUP_EVERY} epochs → {BACKUP_ROOT}")
print(f"  Output       : {OUTPUT_PATH}/{EXPERIMENT}")
print(f"{'='*70}")

# ---------------------------------------------------------------------------
# TRAIN
# ---------------------------------------------------------------------------
results = model.train(
    data=YAML_PATH,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMGSZ,
    device=0,
    workers=2,
    seed=SEED,
    deterministic=True,

    optimizer=OPTIMIZER,
    lr0=LR0,
    lrf=LRF,
    momentum=0.937,
    weight_decay=WEIGHT_DECAY,
    warmup_epochs=WARMUP_EPOCHS,
    warmup_momentum=WARMUP_MOM,
    warmup_bias_lr=WARMUP_BIAS_LR,
    cos_lr=True,

    patience=PATIENCE,

    augment=True,
    hsv_h=HSV_H,
    hsv_s=HSV_S,
    hsv_v=HSV_V,
    degrees=DEGREES,
    translate=TRANSLATE,
    scale=SCALE,
    shear=SHEAR,
    perspective=PERSPECTIVE,
    flipud=FLIPUD,
    fliplr=FLIPLR,
    mosaic=MOSAIC,
    mixup=MIXUP,
    copy_paste=COPY_PASTE,
    close_mosaic=CLOSE_MOSAIC,
    erasing=ERASING,

    box=BOX_LOSS,
    cls=CLS_LOSS,
    dfl=DFL_LOSS,

    label_smoothing=LABEL_SMOOTH,
    dropout=0.1,
    nbs=64,

    amp=True,

    cache='disk',
    rect=False,
    multi_scale=False,

    project=OUTPUT_PATH,
    name=EXPERIMENT,
    exist_ok=True,
    save=True,
    save_period=25,

    verbose=True,
    plots=True,
)

print(f"\n{'='*70}")
print("Training complete.")
print(f"Best weights : {OUTPUT_PATH}/{EXPERIMENT}/weights/best.pt")
print(f"Last weights : {OUTPUT_PATH}/{EXPERIMENT}/weights/last.pt")
print(f"Backups      : {BACKUP_ROOT}/")
print(f"{'='*70}")


## 5. Validation & Metrics

In [ ]:
# ==============================================================================
# 5. VALIDATION
# ==============================================================================

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "ultralytics>=8.1.0", "-q"], check=True)

from ultralytics import YOLO
from pathlib import Path

# ---------------------------------------------------------------------------
# Restore paths/constants (safe to re-run even if cells 3-4 already ran)
# ---------------------------------------------------------------------------
OUTPUT_PATH  = "/kaggle/working/experiments/yolo"
EXPERIMENT   = "mp_yolov8m_single_class"
DATASET_PATH = "/kaggle/working/dataset"
YAML_PATH    = f"{DATASET_PATH}/dataset.yaml"
IMGSZ        = 1280
BATCH_SIZE   = 2
BACKUP_ROOT  = Path("/kaggle/working/backups")

# ---------------------------------------------------------------------------
# Locate best.pt — primary → backup → broad search
# ---------------------------------------------------------------------------
def find_best_weights(primary, backup_root):
    if Path(primary).exists():
        return primary, "primary path"

    # Latest epoch backup
    backups = sorted(backup_root.glob("epoch_*/best.pt")) if backup_root.exists() else []
    if backups:
        return str(backups[-1]), f"backup ({backups[-1].parent.name})"

    # Broad fallback
    candidates = sorted(Path("/kaggle/working").rglob("best.pt"))
    if candidates:
        return str(candidates[-1]), "broad search"

    raise FileNotFoundError(
        "\n\n*** WEIGHTS NOT FOUND ***\n"
        "/kaggle/working/ appears empty. This happens when the kernel is restarted\n"
        "after training — Kaggle wipes /kaggle/working/ on every new session.\n\n"
        "HOW TO RECOVER:\n"
        "  1. Go back to the session where training ran.\n"
        "  2. Click 'Save Version' (top-right) BEFORE the session ends.\n"
        "  3. In a new notebook, click 'Add Data' and attach that saved version.\n"
        "  4. Your weights will appear under /kaggle/input/<your-saved-output>/\n\n"
        "Next time: the auto-backup callback in cell 4 will save weights every\n"
        "50 epochs to /kaggle/working/backups/ — visible in the Output tab live."
    )

BEST_WEIGHTS, source = find_best_weights(
    f"{OUTPUT_PATH}/{EXPERIMENT}/weights/best.pt", BACKUP_ROOT
)
print(f"[OK] Using best.pt from {source}: {BEST_WEIGHTS}")

# ---------------------------------------------------------------------------
# Validate
# ---------------------------------------------------------------------------
model = YOLO(BEST_WEIGHTS)

metrics = model.val(
    data=YAML_PATH,
    imgsz=IMGSZ,
    batch=BATCH_SIZE,
    conf=0.001,
    iou=0.6,
    max_det=1000,
    plots=True,
    save_json=True,
)

print(f"\n{'='*70}")
print("  VALIDATION RESULTS — Single-Class Microplastic Detection")
print(f"{'='*70}")
print(f"  mAP@0.50        : {metrics.box.map50:.4f}")
print(f"  mAP@0.50:0.95   : {metrics.box.map:.4f}")
print(f"  Precision (mean) : {metrics.box.mp:.4f}")
print(f"  Recall (mean)    : {metrics.box.mr:.4f}")
print(f"{'='*70}")

print(f"\n  {'Class':15s}  {'AP@0.50':>8s}  {'AP@0.50:0.95':>12s}")
print(f"  {'-'*15}  {'-'*8}  {'-'*12}")
print(f"  {'microplastic':15s}  {metrics.box.map50:.4f}      {metrics.box.map:.4f}")


## 5.1 Training Curves & Sample Predictions

In [ ]:
# ==============================================================================
# 5.1 TRAINING CURVES & SAMPLE PREDICTIONS
# ==============================================================================

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "ultralytics>=8.1.0", "-q"], check=True)

import matplotlib.pyplot as plt
import cv2
from pathlib import Path
from IPython.display import Image, display
from ultralytics import YOLO

OUTPUT_PATH  = "/kaggle/working/experiments/yolo"
EXPERIMENT   = "mp_yolov8m_single_class"
DATASET_PATH = "/kaggle/working/dataset"
IMGSZ        = 1280
BACKUP_ROOT  = Path("/kaggle/working/backups")

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def find_best_weights(primary, backup_root):
    if Path(primary).exists():
        return str(primary)
    backups = sorted(backup_root.glob("epoch_*/best.pt")) if backup_root.exists() else []
    if backups:
        return str(backups[-1])
    candidates = sorted(Path("/kaggle/working").rglob("best.pt"))
    if candidates:
        return str(candidates[-1])
    raise FileNotFoundError("best.pt not found. Did training complete?")

BEST_WEIGHTS = find_best_weights(
    f"{OUTPUT_PATH}/{EXPERIMENT}/weights/best.pt", BACKUP_ROOT
)
print(f"Using: {BEST_WEIGHTS}")

results_dir = Path(BEST_WEIGHTS).parent.parent

for pf in ["results.png", "confusion_matrix_normalized.png", "PR_curve.png", "F1_curve.png"]:
    plot_path = results_dir / pf
    if plot_path.exists():
        print(f"\n--- {pf} ---")
        display(Image(filename=str(plot_path), width=800))
    else:
        print(f"[SKIP] {pf} not found")

# Filter to image files only — skip .npy and other non-image formats
val_images_dir = Path(DATASET_PATH) / "images" / "val"
sample_images  = sorted(
    p for p in val_images_dir.glob("*") if p.suffix.lower() in IMG_EXTS
)[:6]

if sample_images:
    model = YOLO(BEST_WEIGHTS)
    preds = model(
        [str(p) for p in sample_images],
        imgsz=IMGSZ,
        conf=0.25,
        iou=0.45,
        max_det=500,
    )

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    for ax, result in zip(axes.flatten(), preds):
        img = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(f"{len(result.boxes)} detections", fontsize=11)
        ax.axis("off")

    plt.suptitle("Sample Validation Predictions (conf >= 0.25)", fontsize=14, y=1.01)
    plt.tight_layout()
    save_path = results_dir / "sample_predictions_single_class.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")
else:
    print(f"[WARN] No image files found in {val_images_dir}")
    print(f"  Files present: {[p.name for p in sorted(val_images_dir.glob('*'))[:10]]}")

print("\nVisualisation complete.")


## 6. Export & Save Weights

On Kaggle, all files in `/kaggle/working/` are preserved when you **Save Version**.

### How to Download Your Trained Model

**Option 1 — Direct download:**
1. After notebook finishes, go to the **Output** tab
2. Navigate to `experiments/yolo/mp_yolov8m_single_class/weights/`
3. Download `best.pt`

**Option 2 — Save as Kaggle Dataset (reusable):**
1. Click **Save Version** → **Save & Run All**
2. The entire `/kaggle/working/` becomes a downloadable output
3. You can attach this output as input to other notebooks

**Option 3 — Copy to a convenient location first:**

In [ ]:
# ==============================================================================
# 6. EXPORT & ORGANIZE WEIGHTS
# ==============================================================================

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "ultralytics>=8.1.0", "-q"], check=True)

from ultralytics import YOLO
from pathlib import Path
import shutil

OUTPUT_PATH  = "/kaggle/working/experiments/yolo"
EXPERIMENT   = "mp_yolov8m_single_class"
IMGSZ        = 1280
BACKUP_ROOT  = Path("/kaggle/working/backups")

def find_best_weights(primary, backup_root):
    if Path(primary).exists():
        return str(primary)
    backups = sorted(backup_root.glob("epoch_*/best.pt")) if backup_root.exists() else []
    if backups:
        return str(backups[-1])
    candidates = sorted(Path("/kaggle/working").rglob("best.pt"))
    if candidates:
        return str(candidates[-1])
    raise FileNotFoundError("best.pt not found. Did training complete?")

BEST_WEIGHTS = find_best_weights(
    f"{OUTPUT_PATH}/{EXPERIMENT}/weights/best.pt", BACKUP_ROOT
)
print(f"Using: {BEST_WEIGHTS}")

model = YOLO(BEST_WEIGHTS)

# Export ONNX
onnx_path = model.export(format="onnx", imgsz=IMGSZ, simplify=True)
print(f"ONNX exported : {onnx_path}")

# List all saved weights alongside best.pt
weights_dir = Path(BEST_WEIGHTS).parent
for w in sorted(weights_dir.glob("*.pt")):
    size_mb = w.stat().st_size / 1e6
    print(f"Saved : {w}  ({size_mb:.1f} MB)")

# List available backups
if BACKUP_ROOT.exists():
    print("\nAvailable backups:")
    for b in sorted(BACKUP_ROOT.glob("epoch_*")):
        pts = list(b.glob("*.pt"))
        print(f"  {b.name}  ({len(pts)} weight files)")

# Copy best.pt to root of /kaggle/working/ for easy download
easy_path = Path("/kaggle/working/best.pt")
shutil.copy2(BEST_WEIGHTS, str(easy_path))
print(f"\nCopied best.pt to: {easy_path}")

print(f"\n{'='*70}")
print("HOW TO GET YOUR MODEL:")
print(f"{'='*70}")
print("1. Click 'Save Version' at top right")
print("2. After saving, go to Output tab")
print("3. Download best.pt from the output files")
print("4. Place it at: experiments/yolo/best.pt in your local project")
print(f"{'='*70}")


## Appendix: Resume Training After Session Timeout

If the Kaggle session times out (12-hour max), you can resume:

1. **Save Version** of the notebook first (this preserves `/kaggle/working/`)
2. Create a **new notebook** and add the previous output as a **Dataset**
3. Run the cell below, changing the paths to point to the saved output

In [ ]:
# ==============================================================================
# RESUME TRAINING AFTER SESSION TIMEOUT
# Uncomment all lines below and run if session was interrupted.
# 
# STEPS:
# 1. Save Version of this notebook (preserves output)
# 2. The saved output becomes a dataset you can add to a new notebook
# 3. Add both: your original dataset AND the saved output
# 4. Update PREV_OUTPUT_DATASET below to the saved output name
# ==============================================================================

# !pip install ultralytics>=8.1.0 --quiet
# from ultralytics import YOLO
# import shutil, os, yaml

# # --- Change these to match your Kaggle dataset names ---
# KAGGLE_DATASET_NAME = "mp-yolo-augmented-single"
# PREV_OUTPUT_DATASET = "your-username/your-notebook-name"  # the saved version output

# # Re-copy dataset to working directory
# SRC = f"/kaggle/input/{KAGGLE_DATASET_NAME}"
# DST = "/kaggle/working/dataset"
# if not os.path.exists(DST):
#     shutil.copytree(SRC, DST)
#     yaml_path = f"{DST}/dataset.yaml"
#     with open(yaml_path) as f:
#         cfg = yaml.safe_load(f)
#     cfg['path'] = DST
#     with open(yaml_path, 'w') as f:
#         yaml.dump(cfg, f, default_flow_style=False)

# # Copy previous weights from saved output
# PREV_WEIGHTS = f"/kaggle/input/{PREV_OUTPUT_DATASET}/experiments/yolo/mp_yolov8m_single_class/weights/last.pt"
# model = YOLO(PREV_WEIGHTS)
# results = model.train(resume=True)
# print("Resumed training complete.")